# D-Wave Ocean SDK Framework with Simulated Annealing

This tutorial outlines how to use the D-Wave Ocean SDK to formulate and solve a QUBO optimization problem using simulated annealing. Simulated annealing runs entirely on the local machine, so no access to the D-Wave Leap quantum cloud service is required.

**If you do have access to the Leap quantum cloud service then check out our tutorial on [solving QUBO problems using a quantum annealer](https://kaufmann-group.github.io/qstep-demo/D-wave%20SDK%20Demonstration%20Quantum%20Annealing.html).**

<a class="notebook-download"
   href="./downloads/simulated_annealing_notebook.txt"
   download="D-wave SDK Demonstration Quantum Annealing.ipynb">
  Download notebook ↓
</a>

## Installation & Setup



In this section, we outline how to install the D-Wave Ocean SDK. **All of these steps should be performed in a terminal opened in the directory where you downloaded or saved the notebook.** It is recommended that the Ocean SDK be installed inside a Python virtual environment rather than directly into the system. Virtual environments keep the packages used for this project isolated from other Python projects and avoid potential dependency conflicts. **[Python](https://www.python.org/downloads/) must already be installed on the machine running this demonstration.**

### Installing a Virtual Environment

The virtual environment can also be registered as a Jupyter kernel, allowing the notebook to run directly inside the same environment in which the Ocean SDK is installed.

#### macOS or Linux

To create and activate the virtual environment, run:

```bash
python3 -m venv ocean
source ocean/bin/activate
```

#### Windows Command Prompt

To create and activate the virtual environment, run:

```bat 
python -m venv ocean
ocean\Scripts\activate.bat
```

#### Windows PowerShell

To create and activate the virtual environment, run:

```powershell 
python -m venv ocean
.\ocean\Scripts\Activate.ps1
```

If PowerShell reports that *"running scripts is disabled on this system,"* run:

```powershell 
Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope CurrentUser
```

Then retry the activation command.

### Installing Packages

Once the environment is activated, any Python packages installed with `pip` will be installed inside the ocean environment rather than globally. To install the Ocean SDK and other libraries that will be used run the commands (for any operating system),

``` bash
python -m pip install --upgrade pip

python -m pip install dwave-ocean-sdk
python -m pip install jupyter notebook ipykernel
```

Since this tutorial uses Jupyter notebooks we must register the environment with Jupyter,

``` bash
python -m ipykernel install --user --name ocean --display-name "Ocean Environment"
```

**Now you can open the Jupyter Notebook**. To open Jupyter run, 

``` bash
jupyter notebook
```

After migrating to this notebook within Jupyter **make sure to select the "Ocean Environment" kernel**.

## Review

Recall that quantum annealers are designed to solve certain classes of optimization problems, one of the most common being [Quadratic Unconstrained Binary Optimization (QUBO)](https://en.wikipedia.org/wiki/Quadratic_unconstrained_binary_optimization) problems. D-Wave's Quantum Processing Unit (QPU) represents these problems using an Ising Hamiltonian of the form

$$H(\mathbf{s}) = \sum_i h_i s_i + \sum_{i<j} J_{ij}s_i s_j,\qquad s_i\in\{-1,+1\},$$

where the coefficients $h_i$ represent individual biases and $J_{ij}$ represent interactions between pairs of spins. Using a simple change of variables between Ising spins and binary variables, typically

$$s_i = 2x_i-1,$$

we can rewrite the same problem as a QUBO,

$$Q(\mathbf{x}) = \sum_i a_i x_i + \sum_{i<j} b_{ij}x_i x_j, \qquad x_i\in\{0,1\}.$$

This allows an optimization problem written as a QUBO to be mapped directly onto the Ising model implemented by the D-Wave hardware. The QPU uses a network of superconducting qubits whose interactions are programmed according to the coefficients of the problem. During quantum annealing, the system is driven toward low-energy configurations of the Ising Hamiltonian. After measurement, these spin configurations can be converted back into binary variables, with the lowest-energy solutions corresponding to minima of the original QUBO objective function. Learn more about the physics of quantum annealing in [this]() notebook!

## Optimization Problem Statement: Max-Cut

<div style="display: flex; align-items: center; gap: 30px;">
<div style="flex: 2;">
<p>
A canonical example of a QUBO is the NP-hard <strong>Max-Cut problem</strong>.
Given a graph consisting of vertices connected by edges, the goal is to divide
the vertices into two groups such that the total weight of the edges crossing
between the groups is as large as possible. In other words, we want to find a
partition that maximizes the total weight of the edges crossing the cut.
Max-Cut is a useful example because it has a simple mathematical formulation
while still showing how a combinatorial optimization problem can be mapped onto
a QUBO and solved using the D-Wave framework.
</p>
</div>

<div style="flex: 1; text-align: center;">
<img src="https://raw.githubusercontent.com/kaufmann-group/qstep-demo/main/images/max-cut.png" style="width: 100%; max-width: 260px; height: auto;">
</div>
</div>

We assign each vertex a binary variable,

$$x_i = \begin{cases} 0 & \text{if vertex } i \text{ is in group 0},\\ 1 & \text{if vertex } i \text{ is in group 1}. \end{cases}$$

For an edge connecting vertices $i$ and $j$, the expression

$$x_i+x_j-2x_ix_j$$

is equal to $1$ when the two vertices are in different groups and $0$ when they are in the same group. This can be seen directly from the truth table, which is identical to the XOR truth table:

$$\begin{array}{c|c|c} x_i & x_j & x_i+x_j-2x_ix_j\\ \hline 0 & 0 & 0\\ 0 & 1 & 1\\ 1 & 0 & 1\\ 1 & 1 & 0 \end{array}$$

Therefore, this expression indicates whether a given edge crosses the cut. If each edge has a weight $w_{ij}$, then the total weight of all edges crossing between the two groups is

$$\sum_{(i,j)} w_{ij}\left(x_i+x_j-2x_ix_j\right).$$

The goal of the Max-Cut problem is therefore to maximize this quantity,

$$\max_{\mathbf{x}} \sum_{(i,j)} w_{ij} \left(x_i+x_j-2x_ix_j\right)$$

Because the objective is quadratic in binary variables, it can be written directly as a QUBO. Since QUBOs are conventionally expressed as minimization problems, maximizing the cut weight is equivalent to minimizing its negative:

$$H(\mathbf{x}) = -\sum_{(i,j)} w_{ij} \left(x_i+x_j-2x_ix_j\right)$$

with

$$x_i\in\{0,1\}.$$

The minimum of $H(\mathbf{x})$ therefore corresponds to the partition of the graph with the maximum total edge weight crossing between the two groups.

<div style="display: flex; align-items: center; gap: 30px; flex-wrap: wrap;">
<div style="flex: 2; min-width: 260px;">
<p>
For this demonstration, let's solve the Max-Cut problem for the weighted
graph shown on the right, with edge weights
</p>

$$
\begin{aligned}
(A,B)&=4, & (A,C)&=2, & (B,C)&=1,\\
(B,D)&=5, & (C,D)&=3, & (B,E)&=2,\\
(C,E)&=4, & (D,E)&=3.
\end{aligned}
$$

</div>

<div style="flex: 1; min-width: 220px; text-align: center;">
<img
  src="https://raw.githubusercontent.com/kaufmann-group/qstep-demo/main/images/easy_graph.png"
  style="width: 100%; max-width: 320px; height: auto;">
</div>
</div>

In [ ]:
edges = [("A", "B", 4), ("A", "C", 2), ("B", "C", 1), ("B", "D", 5), ("C", "D", 3), ("B", "E", 2), ("C", "E", 4), ("D", "E", 3)]

## Using D-Wave's Simulated Annealer

The input to the D-Wave system is a QUBO objective matrix in the form of a dictionary where each element represents the coefficient multiplying the $x_ix_j$ term, 

$$Q[(i,j)] = \text{coefficient multiplying } x_i x_j.$$

Using `defaultdict` we break down the expression,

$$-w_{ij}\left(x_i+x_j-2x_ix_j\right)$$

as follows.

In [ ]:
from collections import defaultdict

Q = defaultdict(float)

for i, j, weight in edges:

    # -w_ij * x_i terms
    Q[(i, i)] -= weight

    # -w_ij * x_j terms
    Q[(j, j)] -= weight

    # +2 * w_ij * x_i * x_j terms
    Q[(i, j)] += 2 * weight

Q = dict(Q)

Unlike D-Wave's quantum annealer, simulated annealing runs locally and does not require access to the Leap quantum cloud service. We can create a simulated annealing sampler using `SimulatedAnnealingSampler`.

In [ ]:
from dwave.samplers import SimulatedAnnealingSampler

sampler = SimulatedAnnealingSampler()

Next the QUBO is directly passed into the sampler. The `num_reads` parameter specifies how many times the simulated annealing process is run, with the lowest-energy solution returned as the best solution found and is exactaly analogous to the quantum annealer.  

In [ ]:
sampleset = sampler.sample_qubo(Q, num_reads=1000)

best = sampleset.first # returns lowest energy sample in the SampleSet
solution = dict(best.sample)  

print("Solution:", solution)
print("Energy:", best.energy)

Recall that `solution` is still in its binary representation, where each vertex has been assigned either `0` or `1`; so it follows to separate the vertices into the two groups.

In [ ]:
group_0 = [vertex for vertex, value in solution.items() if value == 0]
group_1 = [vertex for vertex, value in solution.items() if value == 1]

print("Group 0:", group_0)
print("Group 1:", group_1)

# Solve for the max cut weight 
cut_weight = 0

for i, j, weight in edges:
    if solution[i] != solution[j]:
        cut_weight += weight

print("Cut weight:", cut_weight)

## QUBO Practise: Number Partitioning

Suppose we are given the numbers

$$[3,5,6,7,11,12,14,23,26]$$

We want to divide these numbers into two groups such that the difference between the sums of the two groups is as small as possible.

Define

$$ x_i = \begin{cases} 0 & \text{if number } a_i \text{ is placed in Group 0},\\ 1 & \text{if number } a_i \text{ is placed in Group 1}. \end{cases} $$

One way to represent the difference between the two group sums is

$$\sum_i a_i(2x_i-1).$$

**To Do:** Since we want this difference to be as close to zero as possible, construct an objective function that minimizes its magnitude. Use simulated annealing to solve for the solution of this problem!

**Hint.** The difference can be positive or negative, so minimizing it directly
is not what we want. Minimize its square instead:

$$H(\mathbf{x}) = \left(\sum_i a_i(2x_i-1)\right)^2.$$

Expand this into a QUBO. Recall that $x_i^2 = x_i$ for binary variables, so all
squared terms collapse onto the diagonal of $Q$. 

In [ ]:
numbers = [3, 5, 6, 7, 11, 12, 14, 23, 26]
S = sum(numbers)

In [ ]:
Q_number_partitioning = defaultdict(float)

for i in range(len(a)):
    Q_number_partitioning[(i, i)] += 4 * a[i] * (a[i] - S)

for i in range(len(a)):
    for j in range(i + 1, len(a)):
        Q_number_partitioning[(i, j)] += 8 * a[i] * a[j]

In [ ]:
## Solve for the solutions of the QUBO here! 